# MediCitas: demanda por médico y disponibilidad

Este notebook cubre únicamente los puntos 3 y 5. Entrena un modelo supervisado para estimar la demanda futura de cada médico y calcula la disponibilidad como `100 - demanda`. No usa información personal de pacientes.

In [ ]:
from pathlib import Path
from datetime import timedelta
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEMILLA = 42
HORIZONTE_DIAS = 14
VERSION_MODELO = 'rf-demanda-medico-v1'
RUTA_DATOS = Path('/kaggle/input/medicitas-historico/demanda_medicos_dataset.csv')
RUTA_SALIDA = Path('/kaggle/working')
np.random.seed(SEMILLA)

In [ ]:
def generar_datos_sinteticos(dias=365):
    rng = np.random.default_rng(SEMILLA)
    fechas = pd.date_range(end=pd.Timestamp.today().normalize() - timedelta(days=1), periods=dias)
    especialidades = [1, 2, 3, 4, 5, 6, 7, 1]
    clinicas = [1, 1, 2, 2, 3, 3, 4, 4]
    efecto_medico = rng.normal(0, 0.11, 8)
    filas = []
    for fecha in fechas:
        dia_semana = fecha.isoweekday()
        if dia_semana > 5:
            continue
        for medico_id in range(1, 9):
            capacidad = 10 + (8 if medico_id % 2 == 0 and dia_semana in (1, 3, 5) else 0)
            estacionalidad = 0.08 * np.sin(2 * np.pi * fecha.month / 12)
            efecto_dia = {1: 0.13, 2: 0.04, 3: 0.08, 4: 0.02, 5: -0.05}[dia_semana]
            tasa = np.clip(0.48 + efecto_medico[medico_id - 1] + efecto_dia + estacionalidad + rng.normal(0, 0.07), 0.05, 0.98)
            solicitudes = rng.binomial(capacidad, tasa)
            canceladas = rng.binomial(solicitudes, 0.08)
            reservadas = solicitudes - canceladas
            filas.append({
                'medico_id': medico_id, 'especialidad_id': especialidades[medico_id - 1],
                'clinica_id': clinicas[medico_id - 1], 'fecha': fecha,
                'dia_semana': dia_semana, 'mes': fecha.month, 'capacidad': capacidad,
                'reservadas': reservadas, 'canceladas': canceladas,
                'demanda_pct': round(100 * reservadas / capacidad, 2)
            })
    return pd.DataFrame(filas)

if RUTA_DATOS.exists():
    datos = pd.read_csv(RUTA_DATOS)
    ORIGEN_DATOS = 'historico'
else:
    warnings.warn('No se encontró el dataset histórico: se usarán datos sintéticos solo para demostración.')
    datos = generar_datos_sinteticos()
    ORIGEN_DATOS = 'sintetico'

datos['fecha'] = pd.to_datetime(datos['fecha'])
datos = datos.sort_values(['medico_id', 'fecha']).reset_index(drop=True)
datos.head()

In [ ]:
datos['cancelacion_pct'] = 100 * datos['canceladas'] / (datos['reservadas'] + datos['canceladas']).replace(0, np.nan)
datos['demanda_media_28d'] = datos.groupby('medico_id')['demanda_pct'].transform(lambda s: s.shift(1).rolling(28, min_periods=5).mean())
datos['cancelacion_media_28d'] = datos.groupby('medico_id')['cancelacion_pct'].transform(lambda s: s.shift(1).rolling(28, min_periods=5).mean())
datos['demanda_media_28d'] = datos['demanda_media_28d'].fillna(datos.groupby('medico_id')['demanda_pct'].transform('mean')).fillna(datos['demanda_pct'].mean())
datos['cancelacion_media_28d'] = datos['cancelacion_media_28d'].fillna(datos.groupby('medico_id')['cancelacion_pct'].transform('mean')).fillna(0)

columnas_categoricas = ['medico_id', 'especialidad_id', 'clinica_id']
columnas_numericas = ['dia_semana', 'mes', 'capacidad', 'demanda_media_28d', 'cancelacion_media_28d']
columnas_modelo = columnas_categoricas + columnas_numericas
corte = datos['fecha'].quantile(0.80)
entrenamiento = datos[datos['fecha'] <= corte]
prueba = datos[datos['fecha'] > corte]

preprocesador = ColumnTransformer([
    ('categoricas', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
    ('numericas', 'passthrough', columnas_numericas),
])
modelo = Pipeline([
    ('preprocesamiento', preprocesador),
    ('regresor', RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)),
])
modelo.fit(entrenamiento[columnas_modelo], entrenamiento['demanda_pct'])
estimado = np.clip(modelo.predict(prueba[columnas_modelo]), 0, 100)
metricas = {
    'mae': float(mean_absolute_error(prueba['demanda_pct'], estimado)),
    'rmse': float(mean_squared_error(prueba['demanda_pct'], estimado) ** 0.5),
    'r2': float(r2_score(prueba['demanda_pct'], estimado)),
    'origen_datos': ORIGEN_DATOS, 'version_modelo': VERSION_MODELO,
    'filas_entrenamiento': int(len(entrenamiento)), 'filas_prueba': int(len(prueba)),
}
print(json.dumps(metricas, indent=2))
plt.scatter(prueba['demanda_pct'], estimado, alpha=0.25)
plt.xlabel('Demanda real (%)'); plt.ylabel('Demanda estimada (%)'); plt.title('Evaluación del modelo'); plt.show()

In [ ]:
inicio = pd.Timestamp.today().normalize()
fechas_futuras = pd.date_range(inicio, periods=HORIZONTE_DIAS)
medicos = datos[['medico_id', 'especialidad_id', 'clinica_id']].drop_duplicates('medico_id')
ultimos = datos.sort_values('fecha').groupby('medico_id').tail(1).set_index('medico_id')
capacidad_tipo = datos.groupby(['medico_id', 'dia_semana'])['capacidad'].median().to_dict()
filas_futuras = []
for medico in medicos.itertuples(index=False):
    for fecha in fechas_futuras:
        dia = fecha.isoweekday()
        capacidad = int(capacidad_tipo.get((medico.medico_id, dia), 0))
        if capacidad <= 0:
            continue
        ultimo = ultimos.loc[medico.medico_id]
        filas_futuras.append({
            'medico_id': medico.medico_id, 'especialidad_id': medico.especialidad_id,
            'clinica_id': medico.clinica_id, 'fecha': fecha, 'dia_semana': dia,
            'mes': fecha.month, 'capacidad': capacidad,
            'demanda_media_28d': ultimo.demanda_media_28d,
            'cancelacion_media_28d': ultimo.cancelacion_media_28d,
        })
futuro = pd.DataFrame(filas_futuras)
futuro['demanda_dia'] = np.clip(modelo.predict(futuro[columnas_modelo]), 0, 100)
futuro['carga_estimada'] = futuro['capacidad'] * futuro['demanda_dia'] / 100
resumen = futuro.groupby('medico_id', as_index=False).agg(capacidad=('capacidad', 'sum'), carga_estimada=('carga_estimada', 'sum'))
resumen['demanda_estimada'] = (100 * resumen['carga_estimada'] / resumen['capacidad']).clip(0, 100).round(2)
resumen['disponibilidad_estimada'] = (100 - resumen['demanda_estimada']).round(2)
resumen['nivel_demanda'] = pd.cut(resumen['demanda_estimada'], bins=[-0.01, 35, 60, 80, 100], labels=['baja', 'media', 'alta', 'saturada']).astype(str)
resumen['fecha_generacion'] = pd.Timestamp.now(tz='UTC').isoformat()
resumen['fecha_inicio'] = inicio.date().isoformat()
resumen['fecha_fin'] = fechas_futuras[-1].date().isoformat()
resumen['horizonte_dias'] = HORIZONTE_DIAS
resumen['version_modelo'] = VERSION_MODELO
resumen['origen_datos'] = ORIGEN_DATOS
resumen['mae_modelo'] = round(metricas['mae'], 4)
salida = resumen[['medico_id', 'fecha_generacion', 'fecha_inicio', 'fecha_fin', 'horizonte_dias', 'demanda_estimada', 'disponibilidad_estimada', 'nivel_demanda', 'version_modelo', 'origen_datos', 'mae_modelo']]
salida.to_csv(RUTA_SALIDA / 'predicciones_medicos.csv', index=False)
joblib.dump(modelo, RUTA_SALIDA / 'modelo_demanda_medico.joblib')
with open(RUTA_SALIDA / 'metricas_modelo.json', 'w', encoding='utf-8') as archivo:
    json.dump(metricas, archivo, indent=2)
salida